# In-Class Exercise: Sentiment Analysis + Text Classification + Prompt Engineering

**Course:** NLP / Text Mining  
**Total Points:** 30  
**Suggested Time:** about 45 minutes total  
**Dataset:** `sentiment_classification_combined_data.csv`

**Instructions**
- Complete all questions in this notebook.
- Run all cells before submission.
- Keep your answers clear and organized.
- For the prompt engineering part, you do **not** need an API key.


## Part A — Sentiment Analysis with VADER (10 points)
In this part, use **VADER** to compute sentiment scores for the text data.

In [1]:
# Upload the dataset file from your computer (Google Colab)
from google.colab import files

uploaded = files.upload()
filename = next(iter(uploaded))
print("Uploaded file name:", filename)


Saving sentiment_classification_combined_data.csv to sentiment_classification_combined_data.csv
Uploaded file name: sentiment_classification_combined_data.csv


### Q1. Load and inspect the dataset (2 points)
Show:
1. the first 5 rows  
2. the shape of the dataset  
3. the column names

In [4]:
# Read the uploaded CSV file
import pandas as pd

df = pd.read_csv(filename)

# 1. First 5 rows
print("First 5 rows:")
display(df.head())

# 2. Shape
print("Shape:", df.shape)

# 3. Column names
print("Column names:", df.columns.tolist())

First 5 rows:


,Text,TrueLabel,Score
0,I loved this healthcare AI workshop. The examp...,positive,5
1,"This lecture was okay, but some parts were con...",neutral,3
2,The notebook was easy to follow and very pract...,positive,5
3,I did not like the explanation of the model re...,negative,1
4,The activity was interesting and helped me und...,positive,4


Shape: (24, 3)
Column names: ['Text', 'TrueLabel', 'Score']


### Q2. Apply VADER to compute sentiment scores (3 points)
- Import VADER
- Download the lexicon if needed
- Create a new column called `compound`
- Show the first 5 rows of `Text` and `compound`

In [5]:
!pip -q install nltk
import nltk
nltk.download('vader_lexicon')
from nltk.sentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


In [6]:
# Apply VADER to compute compound score for each text
df["compound"] = df["Text"].apply(lambda text: sia.polarity_scores(text)["compound"])

# Show first 5 rows of Text and compound
print(df[["Text", "compound"]].head())

                                                Text  compound
0  I loved this healthcare AI workshop. The examp...    0.8555
1  This lecture was okay, but some parts were con...   -0.2263
2  The notebook was easy to follow and very pract...    0.4404
3  I did not like the explanation of the model re...   -0.2755
4  The activity was interesting and helped me und...    0.4019


### Q3. Create a sentiment label from the compound score (3 points)
Create a new column called `PredictedSentiment` using this rule:
- `compound >= 0.05` → `positive`
- `compound <= -0.05` → `negative`
- otherwise → `neutral`

Then show the first 10 rows of `Text`, `compound`, and `PredictedSentiment`.

In [7]:
# Create PredictedSentiment column based on compound score
def label_sentiment(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["PredictedSentiment"] = df["compound"].apply(label_sentiment)

# Show first 10 rows of Text, compound, and PredictedSentiment
print(df[["Text", "compound", "PredictedSentiment"]].head(10).to_string(index=False))

                                                                     Text  compound PredictedSentiment
 I loved this healthcare AI workshop. The examples were clear and useful.    0.8555           positive
                    This lecture was okay, but some parts were confusing.   -0.2263           negative
                      The notebook was easy to follow and very practical.    0.4404           positive
                     I did not like the explanation of the model results.   -0.2755           negative
The activity was interesting and helped me understand sentiment analysis.    0.4019           positive
          The class was too fast and I could not follow the coding steps.    0.0000            neutral
     The topic is important, but the instructions were not very detailed.    0.1027           positive
                  I really enjoyed the examples and the live coding demo.    0.5563           positive
                     The dataset was messy and the task felt frustrating.

### Q4. Compare review score and sentiment (2 points)
Group by `Score` and compute the **average compound score** for each score value.

In [8]:
# Group by Score and compute average compound score
avg_compound = df.groupby("Score")["compound"].mean().reset_index()
avg_compound.columns = ["Score", "Average Compound Score"]
avg_compound["Average Compound Score"] = avg_compound["Average Compound Score"].round(4)
print(avg_compound.to_string(index=False))

 Score  Average Compound Score
     1                 -0.5070
     2                 -0.2961
     3                 -0.1508
     4                  0.4783
     5                  0.6212


## Part B — Prompt Engineering (10 points)
This part is based on the **Prompt Engineering** notebook. You do **not** need an API key. Focus on writing better prompts using the concepts of **instruction, context, output format, zero-shot, few-shot, and chain-of-thought**.

In [9]:
# Sample texts for the prompt engineering questions
sample_reviews = [
    "The product arrived late and the quality was poor.",
    "Amazing battery life and elegant design.",
    "It is fine for daily use, nothing special."
]
for i, text in enumerate(sample_reviews, start=1):
    print(f"{i}. {text}")

1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.


### Q1. Zero-shot classification prompt (3 points)
Write a **zero-shot prompt** that asks an AI model to classify each review as **positive, negative, or neutral**.

Your prompt should include:
- a clear instruction
- the three labels
- a request for a clean output format

Store your prompt in a Python variable called `zero_shot_prompt`, then print it.

In [10]:
# Zero-shot classification prompt
zero_shot_prompt = """
You are a sentiment classification assistant.

Classify each of the following reviews as exactly one of these labels: positive, negative, or neutral.
Do not explain your answer. Output only the label for each review on a separate line.

Reviews:
1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.

Output format:
1. <label>
2. <label>
3. <label>
"""

print(zero_shot_prompt)


You are a sentiment classification assistant.

Classify each of the following reviews as exactly one of these labels: positive, negative, or neutral.
Do not explain your answer. Output only the label for each review on a separate line.

Reviews:
1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.

Output format:
1. <label>
2. <label>
3. <label>



### Q2. Few-shot prompt improvement (3 points)
Now improve the previous prompt by adding **two examples**.

Requirements:
- Store it in a variable called `few_shot_prompt`
- Include 2 input-output examples
- Ask the model to classify the 3 sample reviews after the examples

In [11]:
# Few-shot prompt with 2 examples
few_shot_prompt = """
You are a sentiment classification assistant.

Classify each review as exactly one of these labels: positive, negative, or neutral.
Do not explain your answer. Output only the label for each review on a separate line.

Here are two examples to guide you:

Example 1:
Review: "The delivery was fast and the item was exactly as described."
Label: positive

Example 2:
Review: "The packaging was damaged and the product stopped working after one day."
Label: negative

Now classify the following reviews:
1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.

Output format:
1. <label>
2. <label>
3. <label>
"""

print(few_shot_prompt)


You are a sentiment classification assistant.

Classify each review as exactly one of these labels: positive, negative, or neutral.
Do not explain your answer. Output only the label for each review on a separate line.

Here are two examples to guide you:

Example 1:
Review: "The delivery was fast and the item was exactly as described."
Label: positive

Example 2:
Review: "The packaging was damaged and the product stopped working after one day."
Label: negative

Now classify the following reviews:
1. The product arrived late and the quality was poor.
2. Amazing battery life and elegant design.
3. It is fine for daily use, nothing special.

Output format:
1. <label>
2. <label>
3. <label>



### Q3. Prompt comparison and refinement (4 points)
Create a short table or DataFrame with **three columns**:
- `PromptType`
- `MainFeature`
- `WhyUseful`

Include these three prompt types:
1. Zero-shot
2. Few-shot
3. Chain-of-thought

Then, in **2–3 sentences**, explain which one you would choose for a more difficult text analysis task and why.

In [12]:
import pandas as pd

# Prompt comparison table
comparison_df = pd.DataFrame({
    "PromptType": ["Zero-shot", "Few-shot", "Chain-of-thought"],
    "MainFeature": [
        "No examples provided; relies entirely on the model instruction",
        "Includes 2+ input-output examples to guide the model",
        "Asks the model to reason step by step before giving an answer"
    ],
    "WhyUseful": [
        "Quick and simple for straightforward tasks with clear labels",
        "Improves accuracy when the task has specific output patterns",
        "Best for complex or ambiguous tasks that require reasoning"
    ]
})

display(comparison_df)

# Explanation
print("""
For a more difficult text analysis task — such as detecting sarcasm or
classifying multi-label sentiment in long reviews — I would choose
chain-of-thought prompting. It forces the model to reason through the
text step by step before committing to a label, which reduces errors
caused by ambiguous phrasing. Few-shot helps with pattern recognition,
but chain-of-thought is more reliable when the task genuinely requires
interpretation rather than pattern matching.
""")

,PromptType,MainFeature,WhyUseful
0,Zero-shot,No examples provided; relies entirely on the m...,Quick and simple for straightforward tasks wit...
1,Few-shot,Includes 2+ input-output examples to guide the...,Improves accuracy when the task has specific o...
2,Chain-of-thought,Asks the model to reason step by step before g...,Best for complex or ambiguous tasks that requi...



For a more difficult text analysis task — such as detecting sarcasm or 
classifying multi-label sentiment in long reviews — I would choose 
chain-of-thought prompting. It forces the model to reason through the 
text step by step before committing to a label, which reduces errors 
caused by ambiguous phrasing. Few-shot helps with pattern recognition, 
but chain-of-thought is more reliable when the task genuinely requires 
interpretation rather than pattern matching.



## Reflection (optional, no extra points)
Briefly explain the difference between:
- sentiment analysis
- text classification
- prompt engineering